# FERRUS LOCUS — Fregate 03
# Pipeline PLY + Image 360° → GLB texturé
## FLOTTE FERRUS | AD MAJOREM GLORIAM IMPERATORIS

---

### Contrat
```
IN/      <- fichier .ply (geometrie Hunyuan3D)
IN_360/  <- image 360 equirectangulaire (jpg/png/hdr)
OUT/     <- decor_locus.glb + rapport_locus.json
```

In [ ]:
# ============================================================
# CELL 01 — INSTALLATION BLENDER 4.2 LTS
# ============================================================
import subprocess, sys, os
from pathlib import Path

BLENDER_VERSION = '4.2.0'
BLENDER_DIR     = f'/content/blender-{BLENDER_VERSION}-linux-x64'
BLENDER_BIN     = f'{BLENDER_DIR}/blender'
BLENDER_URL     = f'https://download.blender.org/release/Blender4.2/blender-{BLENDER_VERSION}-linux-x64.tar.xz'

if not Path(BLENDER_BIN).exists():
    print(f'Telechargement Blender {BLENDER_VERSION}...')
    subprocess.run(['wget', '-q', '--show-progress', BLENDER_URL], check=True)
    print('Extraction...')
    subprocess.run(['tar', '-xf', f'blender-{BLENDER_VERSION}-linux-x64.tar.xz'], check=True)
    subprocess.run(['rm', f'blender-{BLENDER_VERSION}-linux-x64.tar.xz'], check=True)
    print('Extraction terminee.')
else:
    print('Blender deja present, skip telechargement.')

# Injecter dans le PATH de la session
os.environ['PATH'] = BLENDER_DIR + ':' + os.environ['PATH']

result = subprocess.run(['blender', '--version'], capture_output=True, text=True)
print(result.stdout.strip())
print('[LOCUS] Blender 4.2 LTS pret.')

In [ ]:
# ============================================================
# CELL 02 — CONFIGURATION
# ============================================================
import os
from pathlib import Path

# --- Chemins ---
BASE_DIR    = Path('/content')
IN_DIR      = BASE_DIR / '03_FERRUS_LOCUS' / 'IN'
IN_360_DIR  = BASE_DIR / '03_FERRUS_LOCUS' / 'IN_360'
OUT_DIR     = BASE_DIR / '03_FERRUS_LOCUS' / 'OUT'
SCRIPT_PATH = BASE_DIR / '03_FERRUS_LOCUS' / 'codebase' / 'locus_convert.py'

for d in [IN_DIR, IN_360_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Extensions supportees ---
SUPPORTED_PLY = ('.ply',)
SUPPORTED_IMG = ('.jpg', '.jpeg', '.png', '.hdr', '.exr')

# --- Decimation (Option C : auto par defaut) ---
DECIMATION_LEVEL = 'auto'   # 'auto' | 'none' | 'high' | 'medium' | 'low'
BAKE_RESOLUTION  = 2048     # 1024 pour Colab si memoire limitee

# --- Scan des inputs ---
ply_files = sorted([f for f in IN_DIR.iterdir() if f.suffix.lower() in SUPPORTED_PLY])
img_files = sorted([f for f in IN_360_DIR.iterdir() if f.suffix.lower() in SUPPORTED_IMG])

print(f'PLY trouves  : {[f.name for f in ply_files]}')
print(f'IMG360 trouves: {[f.name for f in img_files]}')

if not ply_files:
    raise FileNotFoundError('Aucun .ply dans IN/ — deposer le fichier Hunyuan3D')
if not img_files:
    raise FileNotFoundError('Aucune image 360 dans IN_360/ — deposer limage equirectangulaire')

PLY_PATH   = str(ply_files[0])
IMG360_PATH= str(img_files[0])
OUTPUT_GLB = str(OUT_DIR / 'decor_locus.glb')

print(f'\nInput PLY  : {PLY_PATH}')
print(f'Input 360  : {IMG360_PATH}')
print(f'Output GLB : {OUTPUT_GLB}')

In [ ]:
# ============================================================
# CELL 02.5 — IMPORT FICHIERS DEPUIS GOOGLE DRIVE (optionnel)
# Executer seulement si les fichiers sont sur Drive
# ============================================================
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/drive')

# Scan automatique Drive pour trouver .ply et images 360
drive_root = Path('/drive/MyDrive')
ply_found = sorted(drive_root.rglob('*.ply'))
img_found = sorted([
    f for f in drive_root.rglob('*')
    if f.suffix.lower() in ('.png', '.jpg', '.jpeg', '.hdr', '.exr')
    and 'FERRUS' in str(f)
])

print('PLY trouves sur Drive:')
for f in ply_found: print(f'  {f}')
print('Images 360 trouvees:')
for f in img_found: print(f'  {f}')

# --- Modifier les chemins ci-dessous selon vos fichiers ---
# PLY_SRC = '/drive/MyDrive/...'
# IMG_SRC = '/drive/MyDrive/...'
# shutil.copy(PLY_SRC,  str(IN_DIR))
# shutil.copy(IMG_SRC,  str(IN_360_DIR))
# print('Copie OK.')

In [ ]:
# ============================================================
# CELL 03.5 — POISSON RECONSTRUCTION (Point cloud → Mesh)
# Requis si le PLY Hunyuan3D est un point cloud (0 faces)
# Produit locus_mesh.ply et met a jour PLY_PATH
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'open3d'], check=True)

import open3d as o3d
import numpy as np
from pathlib import Path

PLY_OUT = str(IN_DIR / 'locus_mesh.ply')

print('Chargement point cloud...')
pcd = o3d.io.read_point_cloud(PLY_PATH)
print(f'Points: {len(pcd.points):,}')

# Verifier si c'est bien un point cloud
if len(np.asarray(pcd.triangles if hasattr(pcd, "triangles") else [])) > 0:
    print('Mesh detecte — pas de reconstruction necessaire.')
else:
    print('Estimation normales...')
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
    )
    pcd.orient_normals_consistent_tangent_plane(100)

    print('Poisson surface reconstruction (depth=8)...')
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8)

    # Supprimer artefacts de bord (5% densite la plus faible)
    threshold = np.percentile(np.asarray(densities), 5)
    mesh.remove_vertices_by_mask(np.asarray(densities) < threshold)
    mesh.compute_vertex_normals()

    print(f'Triangles: {len(mesh.triangles):,}')
    o3d.io.write_triangle_mesh(PLY_OUT, mesh)
    print(f'Mesh sauve: {PLY_OUT}')

    # Mettre a jour PLY_PATH pour Cell 04
    PLY_PATH = PLY_OUT
    print(f'PLY_PATH mis a jour -> {PLY_PATH}')

In [ ]:
# ============================================================
# CELL 03 — CLONE DU REPO (si pas deja present)
# ============================================================
import subprocess

REPO_URL = 'https://github.com/kioka8877-ux/FLOTTE-FERRUS.git'
CLONE_DIR = '/content/03_FERRUS_LOCUS'

if not Path(SCRIPT_PATH).exists():
    print('Clone du repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/FLOTTE-FERRUS'], check=True)
    subprocess.run(['cp', '-r', '/content/FLOTTE-FERRUS/03_FERRUS_LOCUS', '/content/'], check=True)
    print('Clone OK.')
else:
    print('Script deja present, pas de clone necessaire.')

print(f'Script: {SCRIPT_PATH} | Existe: {Path(SCRIPT_PATH).exists()}')

In [ ]:
# ============================================================
# CELL 04 — LANCEMENT PIPELINE LOCUS
# ============================================================
import subprocess, time

cmd = [
    'blender',
    '--background',
    '--python', str(SCRIPT_PATH),
    '--',
    '--ply',      PLY_PATH,
    '--img360',   IMG360_PATH,
    '--output',   OUTPUT_GLB,
    '--decim',    DECIMATION_LEVEL,
    '--bake-res', str(BAKE_RESOLUTION),
]

print('Commande:', ' '.join(cmd))
print('Lancement LOCUS...')
t0 = time.time()

result = subprocess.run(cmd, capture_output=True, text=True)

elapsed = round(time.time() - t0, 1)
print(f'Duree: {elapsed}s')
print('--- STDOUT ---')
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.stderr:
    print('--- STDERR ---')
    print(result.stderr[-1000:])

if result.returncode != 0:
    raise RuntimeError(f'LOCUS ECHEC (code {result.returncode})')
print('LOCUS PIPELINE: SUCCESS')

In [ ]:
# ============================================================
# CELL 05 — RAPPORT + VALIDATION
# ============================================================
import json
from pathlib import Path

rapport_path = Path(OUTPUT_GLB).parent / 'rapport_locus.json'
glb_path     = Path(OUTPUT_GLB)

if rapport_path.exists():
    with open(rapport_path) as f:
        rapport = json.load(f)
    print('=== RAPPORT LOCUS ===')
    for k, v in rapport.items():
        print(f'  {k}: {v}')
else:
    print('Rapport introuvable.')

print()
if glb_path.exists():
    size_mb = glb_path.stat().st_size / (1024*1024)
    print(f'GLB produit : {glb_path.name} ({size_mb:.2f} Mo)')
    print('STATUS: SUCCESS — Decor pret pour integration.')
else:
    print('ERREUR: GLB absent.')

In [ ]:
# ============================================================
# CELL 06 — TELECHARGEMENT DU RESULTAT
# ============================================================
from google.colab import files
from pathlib import Path

glb_path     = Path(OUTPUT_GLB)
rapport_path = glb_path.parent / 'rapport_locus.json'

if glb_path.exists():
    print(f'Telechargement: {glb_path.name}')
    files.download(str(glb_path))

if rapport_path.exists():
    print(f'Telechargement: {rapport_path.name}')
    files.download(str(rapport_path))

print('Tous les fichiers proposes au telechargement.')